# Chạy tổng hợp CHỈ VAL: Tầng 1 → Tầng 2 → một bảng

1. Runtime **GPU** (L4 hoặc T4). Trong 🔑 Secrets bật `GH_TOKEN` cho notebook này.
2. Chạy lần lượt cả 3 cell: Setup (giải nén ảnh khoảng 10 phút), cell I (nhập zip mới trong `MyDrive/Sym/upload/`, không có zip thì bỏ qua), cell L (evaluate val + Tầng 2 + bảng).
3. **Lần đầu:** đổi tên `MyDrive/Sym/runs_aug1` (bản Colab cũ) thành `runs_aug1_colab` trước khi chạy.

Không train, **không đụng test**. Số trên val có λ và ngưỡng chọn trên chính val: chỉ xem chiều hướng, không báo cáo, không so với 63.0. Có thêm run thì chạy lại cả notebook: phần đã làm được bỏ qua.

In [ ]:
import os, sys, json, time, base64, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Letri6102/Thesis_Sym.git'
BRANCH   = 'main'                      # nhánh stage1-ema-valchecks đã merge vào main (21/11)
REPO     = Path('/content/Thesis_Sym')
CODE     = REPO / 'Contrastive_V1'
SYM      = Path('/content/drive/MyDrive/Sym')
SPLITS   = SYM / 'splits_v2'           # manifest mới; splits/ cũ chỉ cho runs_retry1
RUNS     = SYM / 'runs_aug1'           # run mới với strong_train_aug
LOGS     = SYM / 'logs'
IMAGES   = True                        # False: không giải nén ảnh (phiên CPU chỉ chạy Tầng 2, cell E)

tok  = userdata.get('GH_TOKEN')
auth = 'http.extraHeader=Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
def git(*args, cwd=None):
    r = subprocess.run(['git', '-c', auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if r.returncode:
        raise RuntimeError(f'git {args[0]} lỗi (exit {r.returncode}): {r.stderr.strip()}')
try:
    if (REPO / '.git').exists():
        # refspec đầy đủ: bản clone 1 nhánh vẫn đổi được sang nhánh khác (fetch tên trơn chỉ cập nhật FETCH_HEAD)
        git('fetch', '-q', '--depth', '1', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}', cwd=REPO)
        git('reset', '-q', '--hard', f'origin/{BRANCH}', cwd=REPO)
    else:
        git('clone', '-q', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO))
finally:
    del tok, auth
print('commit:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], cwd=REPO,
                                capture_output=True, text=True).stdout.strip())

SHARDS = SYM / 'DISFA_img'                     # 27 tar theo subject, do cell A tạo
if IMAGES and (SHARDS / 'DONE.json').exists():
    DATA = Path('/content/DISFA')
    if not (DATA / '.img_ok').exists():        # chưa giải nén, hoặc lần trước giải nén dở
        t0, meta = time.time(), json.loads((SHARDS / 'DONE.json').read_text())
        DATA.mkdir(parents=True, exist_ok=True)
        print(f'Giải nén {len(meta["subjects"])} tar ({meta["bytes"] / 1e9:.1f} GB) về /content:', flush=True)
        for i, s in enumerate(meta['subjects'], 1):
            subprocess.run(['tar', '-xf', str(SHARDS / f'{s}.tar'), '-C', str(DATA)], check=True, timeout=1800)
            print(f'  {i:>2}/{len(meta["subjects"])} {s}  {(time.time() - t0) / 60:.1f} phút', flush=True)
        n = sum(len(f) for _, _, f in os.walk(DATA / 'img'))
        assert n == meta['n_images'], f'Giải nén được {n} ảnh, cần {meta["n_images"]}'
        shutil.copytree(SYM / 'DISFA' / 'list', DATA / 'list', dirs_exist_ok=True)
        (DATA / '.img_ok').touch()
        print(f'Đã giải nén {n} ảnh về /content trong {time.time() - t0:.0f}s')
else:
    DATA = SYM / 'DISFA'                 # đọc thẳng Drive: chậm và hay lỗi FileNotFoundError
print('DATA =', DATA)

os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
(CODE / 'checkpoints').mkdir(exist_ok=True)
w_src = SYM / 'Contrastive_V1' / 'checkpoints' / 'resnet50-19c8e357.pth'
w_dst = CODE / 'checkpoints' / 'resnet50-19c8e357.pth'
if w_src.exists() and not w_dst.exists():
    shutil.copy(w_src, w_dst)
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CPU', os.cpu_count(),
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

import re
def run_logged(cmd, name):
    """Chạy lệnh con, ghi toàn bộ output vào MyDrive/Sym/logs/ và in các dòng đáng xem.
    (Colab không hiện stdout của tiến trình con nếu chỉ dùng subprocess.run.)"""
    LOGS.mkdir(exist_ok=True)
    log = LOGS / f'{time.strftime("%Y%m%d_%H%M%S")}_{name}.log'
    print(f'>> {name} | log: {log.name}', flush=True)
    with open(log, 'w') as f, subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                               stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line); f.flush()
            if line.startswith('{"epoch"'):
                r = json.loads(line)
                print(f"   ep{r['epoch']:>2}  wa={r['train']['wa']:.4f}  valF1={r['val']['macro_f1']:.3f}"
                      f"  {r['seconds'] / 60:.0f} phút", flush=True)
            elif line.startswith('epoch ') and ' step ' in line:
                if re.search(r'step (1000|2000)/', line):
                    print('   ' + line.rstrip(), flush=True)
            elif 'FutureWarning' not in line and 'warnings.warn' not in line:
                print('   ' + line.rstrip(), flush=True)
    if p.returncode:
        raise RuntimeError(f'{name} dừng với mã {p.returncode}, xem log {log}')

## Cell I — nhập zip mới từ máy 5090

In [ ]:
# ==== Cell I: nhập run train trên máy GPU riêng (MyDrive/Sym/upload/*.zip) — kiểm khoá, không ghi đè ====
import re, zipfile
sys.path.insert(0, str(CODE))
from stage2_decoder.contrast_ci import LOCK, config_problems, run_problems
from refit import run as R

UP, BUSY_MIN = SYM / 'upload', 15
REFIT = SYM / 'runs_refit1'
assert RUNS.name == 'runs_aug1', f'RUNS phải là runs_aug1, đang là {RUNS}'
REFIT_HASH = R.write_manifests(SPLITS, DATA, REFIT)      # manifest refit chuẩn, tính lại từ splits_v2 đã khoá

def ok_aug1(run, v, k, s):
    return not run_problems(run, v, k, s, LOCK) and (run / 'last.pt').exists()

def ok_refit1(run, v, k, s):
    try:
        rec, cfg = (json.loads((run / n).read_text()) for n in ('refit.json', 'config.json'))
    except (OSError, ValueError):
        return False
    a = RUNS / v / f'fold{k}' / f'seed{s}'
    lock = dict(LOCK, manifests={**LOCK['manifests'], k: REFIT_HASH[k]})
    return (rec['manifest_hash'] == REFIT_HASH[k] and R.n_epochs(run) == rec['E'] and (run / 'last.pt').exists()
            and not config_problems(cfg, v, k, s, lock) and ok_aug1(a, v, k, s) and R.selected_epoch(a) == rec['E'])

KINDS = {'aug1': (RUNS, ok_aug1), 'refit1': (REFIT, ok_refit1)}

def park(z, sub):
    (UP / sub).mkdir(parents=True, exist_ok=True)
    shutil.move(str(z), str(UP / sub / z.name))

zips = sorted(UP.glob('*.zip'))                           # aug1_* đứng trước refit1_*: refit cần run (a) đã có
print(f'{len(zips)} zip chờ nhập trong {UP}', flush=True)
for z in zips:
    m = re.fullmatch(r'(aug1|refit1)_(baseline|full)_fold([123])_seed([012])', z.stem)
    if not m:
        print(f'  {z.name}: tên lạ, bỏ qua'); continue
    kind, v, k, s = m[1], m[2], int(m[3]), int(m[4])
    base, ok = KINDS[kind]
    target, prefix = base / v / f'fold{k}' / f'seed{s}', f'{v}/fold{k}/seed{s}/'
    work = Path('/content/pc_import') / z.stem
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    subprocess.run(['cp', str(z), str(work / 'run.zip')], check=True, timeout=1800)     # đọc Drive có timeout
    with zipfile.ZipFile(work / 'run.zip') as f:
        if any(not n.startswith(prefix) or '..' in n.split('/') for n in f.namelist()):
            print(f'  {z.name}: zip chứa đường dẫn lạ → KHÔNG nhập'); park(z, 'rejected'); continue
        f.extractall(work / 'x')
    run = work / 'x' / v / f'fold{k}' / f'seed{s}'
    if not ok(run, v, k, s):
        print(f'  {z.name}: lệch khoá, chưa xong, hoặc thiếu run (a) tương ứng → KHÔNG nhập'); park(z, 'rejected'); continue
    gpu = json.loads((run / 'environment.json').read_text()).get('gpu')
    mixed = sorted({json.loads(p.read_text()).get('gpu') or '?' for b in (RUNS, REFIT)
                    for p in b.glob('*/fold*/seed*/environment.json') if p.parent != target} - {gpu})
    if mixed:                                             # 23/11: cả 18 run (a) + refit phải train trên cùng một máy
        print(f'  {z.name}: train trên {gpu}, nhưng Drive đã có run train trên {mixed} → để lại. '
              f'Dừng cell G/K, đổi tên {RUNS.name} và {REFIT.name} cũ thành *_colab, rồi chạy lại cell'); continue
    if (target / 'config.json').exists():
        if ok(target, v, k, s):
            print(f'  {z.name}: Drive đã có run này hoàn chỉnh → giữ bản Drive, không ghi đè'); park(z, 'duplicate'); continue
        files = [p for p in target.rglob('*') if p.is_file()]
        age = (time.time() - max(p.stat().st_mtime for p in files)) / 60 if files else 1e9
        if age < BUSY_MIN:
            print(f'  {z.name}: run này vừa được ghi {age:.0f} phút trước (Colab có thể đang train) → để lại, chạy lại cell sau'); continue
        dest = SYM / 'failed_runs' / f'{kind}_{v}_fold{k}_seed{s}_{time.strftime("%Y%m%d_%H%M%S")}_truoc_nhap'
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(target), str(dest))
        print(f'  bản dở trên Drive → failed_runs/{dest.name}')
    partial = target.with_name(target.name + '.partial')
    shutil.rmtree(partial, ignore_errors=True)
    partial.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', str(run), str(partial)], check=True, timeout=3600)
    partial.rename(target)
    assert ok(target, v, k, s), f'{z.name}: sau khi chép lên Drive lại lệch, gửi Claude xem'
    park(z, 'done')
    shutil.rmtree(work)
    print(f'  {z.name}: ĐÃ NHẬP (GPU {gpu})', flush=True)

for kind, (base, ok) in KINDS.items():
    n = sum(ok(base / v / f'fold{k}' / f'seed{s}', v, k, s) for v in ('baseline', 'full') for k in (1, 2, 3) for s in (0, 1, 2))
    print(f'{base.name}: {n}/18 run hoàn chỉnh đúng khoá')
print('Đang chờ Drive đồng bộ...', flush=True)
drive.flush_and_unmount()

## Cell L — evaluate val, Tầng 2, bảng tổng hợp

In [ ]:
# ==== Cell L: CHẠY TỔNG HỢP — run Tầng 1 đã có → evaluate val → Tầng 2 → một bảng. CHỈ VAL, không đụng test ====
if not os.path.isdir('/content/drive/MyDrive'):            # cell I tháo Drive ở cuối
    drive.mount('/content/drive')
assert str(DATA).startswith('/content/'), 'evaluate val cần ảnh trong /content: chạy Setup với IMAGES = True'
assert RUNS.name == 'runs_aug1', f'RUNS phải là runs_aug1, đang là {RUNS}'
gpus = {json.loads(p.read_text()).get('gpu') or '?' for p in RUNS.glob('*/fold*/seed*/environment.json')}
assert len(gpus) <= 1, (f'runs_aug1 có run từ nhiều máy {sorted(gpus)}: đổi tên runs_aug1 cũ (Colab) thành '
                        'runs_aug1_colab, rồi nhập lại zip của máy 5090 bằng cell I')
STAGE2 = SYM / 'stage2_v2'
run_logged([sys.executable, '-u', '-m', 'stage2_decoder.run_all', '--runs', RUNS, '--splits', SPLITS,
            '--data-root', DATA, '--output', STAGE2, '--variants', 'baseline', 'full',
            '--folds', 1, 2, 3, '--seeds', 0, 1, 2, '--val-only'], 'tong_hop_stage2_val')
run_logged([sys.executable, '-u', '-m', 'stage2_decoder.val_overview', '--runs', RUNS,
            '--results', STAGE2 / 'results', '--output', STAGE2 / 'val_overview.json'], 'tong_hop_bang_val')
print('Bảng đã ghi vào', STAGE2 / 'val_overview.json', flush=True)
drive.flush_and_unmount()